In [1]:
# Cell 1: latest summary row per run, side by side
import json
import numpy as np
import pandas as pd
from geoscholar import config
from geoscholar.evaluation import bootstrap_ci

pd.set_option("display.width", 200)
RES = config.EVAL_DIR / "results"
summary = pd.read_csv(RES / "answer_summary.csv")
latest = summary.sort_values("run_at").groupby("tag").tail(1).set_index("tag")
cols = ["k", "order", "refusal_rate_unanswerable", "false_refusal_rate", "answered_without_evidence",
        "faithfulness", "citation_coverage", "cites_relevant", "relevance_full", "p50_total_ms"]
print(latest[cols].round(3).T)

tag                         baseline baseline_qwen         k3         k8
k                                  5             5          3          8
order                      relevance     relevance  relevance  relevance
refusal_rate_unanswerable        1.0           1.0        1.0        1.0
false_refusal_rate             0.011         0.011      0.023      0.022
answered_without_evidence        NaN           NaN        0.5        NaN
faithfulness                   0.886         0.855      0.888      0.906
citation_coverage              0.802         0.802      0.789        0.8
cites_relevant                 0.978         0.978        1.0      0.978
relevance_full                 0.778           0.6      0.805      0.764
p50_total_ms                  4453.8        4453.8     4490.3     5205.9


In [3]:
# Cell 2: paired comparison of faithfulness between two runs
def load(tag):
    with open(RES / f"judged_{tag}.jsonl", encoding="utf-8") as f:
        return {r["id"]: r for r in map(json.loads, f)}

def answer_faithfulness(r):
    """Share of one answer's claims that are supported (None if refused or no claims)."""
    v = [c["verdict"] for c in r.get("claims", []) if c["verdict"] != "error"]
    return None if r["refused"] or not v else sum(x == "supported" for x in v) / len(v)

def compare(tag_a, tag_b):
    a, b = load(tag_a), load(tag_b)
    diffs = [answer_faithfulness(a[q]) - answer_faithfulness(b[q]) for q in a
             if q in b and answer_faithfulness(a[q]) is not None and answer_faithfulness(b[q]) is not None]
    lo, hi = bootstrap_ci(diffs)
    verdict = "real difference" if lo > 0 or hi < 0 else "could be noise"
    print(f"faithfulness {tag_a} vs {tag_b}: {np.mean(diffs):+.3f} over {len(diffs)} answers, "
          f"95% CI [{lo:+.3f}, {hi:+.3f}] -> {verdict}")

compare("sandwich", "baseline")
compare("k3", "baseline")
compare("k8", "baseline")

faithfulness sandwich vs baseline: +0.000 over 74 answers, 95% CI [-0.018, +0.019] -> could be noise
faithfulness k3 vs baseline: -0.036 over 87 answers, 95% CI [-0.078, +0.003] -> could be noise
faithfulness k8 vs baseline: -0.002 over 88 answers, 95% CI [-0.049, +0.037] -> could be noise


In [5]:
# Cell 3: which unanswerable questions slipped through?
base = load("baseline")
by_tag = {}
for r in base.values():
    if not r["paper_ids"]:
        for t in r.get("tags") or ["untagged"]:
            by_tag.setdefault(t, []).append(r["refused"])
for t, v in sorted(by_tag.items()):
    print(f"{t:15s} refused {sum(v)}/{len(v)}")

for r in base.values():
    if not r["paper_ids"] and not r["refused"]:
        print("\nQ:", r["question"], "\nA:", r["answer_display"])

false-premise   refused 1/1
in-domain       refused 1/1
off-topic       refused 1/1
untagged        refused 5/5


In [7]:
# Cell 4: median and 95th-percentile time per stage
for tag in ["baseline", "sandwich", "k3", "k8"]:
    t = pd.DataFrame([r["timings_ms"] for r in load(tag).values()])
    print(f"\n{tag}  (ms)")
    print(t.describe(percentiles=[0.5, 0.95]).loc[["50%", "95%"]].round(0))


baseline  (ms)
     retrieve  rerank  generate   total
50%      49.0   372.0    4069.0  4454.0
95%      83.0   411.0    7373.0  7840.0

sandwich  (ms)
     retrieve  rerank  generate   total
50%      50.0   380.0    4365.0  4787.0
95%      78.0   420.0    7782.0  8238.0

k3  (ms)
     retrieve  rerank  generate   total
50%      45.0   362.0    4083.0  4490.0
95%      81.0   403.0    6687.0  7083.0

k8  (ms)
     retrieve  rerank  generate    total
50%      46.0   371.0    4762.0   5206.0
95%      80.0   429.0    9973.0  10365.0
